# 1. Quick Start

pEYES turns raw gaze samples (`t`, `x`, `y`) into classified eye-movement events (fixations, saccades, ...) and lets
you evaluate that classification against human-annotated ground truth. This notebook runs the whole pipeline once,
end to end, so you have a working example before the rest of the guide goes into each step in depth.

**Pipeline:** raw samples &rarr; per-sample **labels** (detection) &rarr; **events** &rarr; summary table.

## Step 1: Load one trial of eye-tracking data

pEYES ships with human-annotated datasets you can download directly (see notebook 2). We'll use one short trial
from the `lund2013` dataset (Andersson et al., 2017): a ~5.6-second video-viewing trial with a mix of fixations,
saccades, and a blink, annotated by a human rater (`RA`).

In [1]:
import numpy as np
import peyes

dataset = peyes.datasets.lund2013(directory="data", save=True, verbose=True)
trial = dataset[dataset["trial_id"] == 51].reset_index(drop=True)

t, x, y, pupil = trial["t"].values, trial["x"].values, trial["y"].values, trial["pupil"].values
pixel_size, viewer_distance = trial["pixel_size"].values[0], trial["viewer_distance"].values[0]
ground_truth = trial["RA"].values  # human-annotated labels

print(f"{len(t)} samples, {t[-1] / 1000:.1f} seconds")

2784 samples, 5.6 seconds


Loading and slicing a dataset like this is the same handful of lines in every notebook, so the rest of this
guide uses a small helper, `_helpers.load_example_trial()`, that wraps it (see [`_helpers.py`](./_helpers.py)).
It returns the same data, plus a `raters` dict of any human-annotated label columns found on the trial:

In [2]:
import _helpers

d = _helpers.load_example_trial()  # same trial as above: lund2013, trial_id=51
assert np.array_equal(d["t"], t) and np.array_equal(d["raters"]["RA"], ground_truth)
d.keys(), d["raters"].keys()

(dict_keys(['t', 'x', 'y', 'pupil', 'pixel_size', 'viewer_distance', 'raters']),
 dict_keys(['RA']))

## Step 2: Detect events with the Engbert algorithm

A **detector** classifies each sample into an `EventLabelEnum` (`UNDEFINED`, `FIXATION`, `SACCADE`, `PSO`,
`SMOOTH_PURSUIT`, `BLINK`). We build one with `peyes.create_detector(algorithm, ...)` — here using the
velocity-based algorithm by Engbert & Kliegl (2003) — then run it with `.detect()`.

In [3]:
detector = peyes.create_detector(
    "engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0,
)
labels, detection_info = detector.detect(
    t=t, x=x, y=y, pixel_size_cm=pixel_size, viewer_distance_cm=viewer_distance,
)
print(labels[:10])
print(detection_info)

[<EventLabelEnum.UNDEFINED: 0>, <EventLabelEnum.UNDEFINED: 0>, <EventLabelEnum.UNDEFINED: 0>, <EventLabelEnum.FIXATION: 1>, <EventLabelEnum.FIXATION: 1>, <EventLabelEnum.FIXATION: 1>, <EventLabelEnum.FIXATION: 1>, <EventLabelEnum.FIXATION: 1>, <EventLabelEnum.FIXATION: 1>, <EventLabelEnum.FIXATION: 1>]
{'x_threshold_velocity_pxs': 2386.6394050488184, 'y_threshold_velocity_px': 1428.0515476430048, 'sampling_rate': np.float64(499.9), 'pixel_size': np.float64(0.03782412011534439), 'viewer_distance': np.float64(67.0), 'runtime': 0.03658723831176758}


`labels` is one `EventLabelEnum` value per sample. `peyes.parse_label` converts between labels' names, values,
and `EventLabelEnum` instances — useful for readable output:

In [4]:
[peyes.parse_label(lbl).name for lbl in labels[:10]]

['UNDEFINED',
 'UNDEFINED',
 'UNDEFINED',
 'FIXATION',
 'FIXATION',
 'FIXATION',
 'FIXATION',
 'FIXATION',
 'FIXATION',
 'FIXATION']

## Step 3: Turn labels into Event objects

A per-sample label is a classification; an **Event** groups the consecutive samples sharing a label into an object
with derived properties (duration, amplitude, peak velocity, ...). Build them with `peyes.create_events`:

In [5]:
events = peyes.create_events(
    labels=labels, t=t, x=x, y=y, pupil=pupil, pixel_size=pixel_size, viewer_distance=viewer_distance,
)
first_event = events[0]
print(f"{first_event.label.name}: {first_event.start_time:.1f}-{first_event.end_time:.1f} ms "
      f"(duration {first_event.duration:.1f} ms)")

FIXATION: 6.0-210.0 ms (duration 204.0 ms)


## Step 4: Summarize events into a table

`peyes.summarize_events` collects every event's features into one `pandas.DataFrame`, one row per event:

In [6]:
events_table = peyes.summarize_events(events)
events_table.head(10)

,event_type,label,start_time,end_time,duration,distance,amplitude,azimuth,peak_velocity,median_velocity,...,start_x,start_y,end_x,end_y,center_pixel,pixel_std,dispersion,ellipse_area,is_outlier,outlier_reasons
0,FIXATION,1,6.000,210.046,204.046,7.356845,0.237962,99.967557,42.993417,13.650236,...,522.4870,423.0247,521.2136,415.7789,"(520.6592281553399, 418.31843883495134)","(1.3781321854123456, 2.85209201717164)",0.522272,0.049609,False,[]
1,SACCADE,2,212.047,272.057,60.010,241.146818,7.788051,178.513830,158.498845,79.263708,...,522.7053,416.1030,281.6396,409.8487,"(347.81459677419355, 412.33902903225817)","(85.4484891508158, 3.520229042144439)",8.706070,2.715996,False,[]
2,FIXATION,1,274.055,496.099,222.044,40.102058,1.297074,6.302341,43.679851,13.020898,...,278.8346,410.6481,318.6943,406.2459,"(291.6303732142857, 410.28303482142854)","(12.000146907694374, 1.5809510192619272)",1.601033,0.265221,False,[]
3,SACCADE,2,498.103,522.111,24.008,127.840747,4.133306,12.796403,148.080598,111.109023,...,319.6409,405.4971,444.3065,377.1820,"(381.0220769230769, 388.2436153846154)","(48.09015486131078, 14.103464151043285)",5.139701,3.510692,False,[]
4,FIXATION,1,524.112,528.115,4.003,2.411057,0.077987,194.343443,26.145356,19.518542,...,445.7363,378.2661,443.4004,378.8634,"(444.44613333333336, 378.6637)","(0.9691638813373706, 0.2811466165544314)",0.094876,0.001146,True,[min_duration]
5,SACCADE,2,530.112,534.112,4.000,5.273568,0.170577,197.506920,45.315013,40.785199,...,441.5088,379.7703,436.4795,381.3567,"(438.9026333333333, 380.62416666666667)","(2.057278221880132, 0.6533031829777594)",0.213989,0.006556,True,[min_duration]
6,FIXATION,1,536.112,542.110,5.998,2.191096,0.070873,197.182899,17.203618,17.179334,...,433.9817,382.3015,431.8884,382.9488,"(432.74295, 382.657425)","(0.8059122362267493, 0.3629671567442555)",0.093314,0.001362,True,[min_duration]
7,SACCADE,2,544.112,572.120,28.008,34.249128,1.107778,356.151584,74.872314,38.087965,...,433.7982,381.4550,467.9701,383.7537,"(452.6205333333334, 383.62452666666667)","(10.597112155278692, 1.7278169476603162)",1.279730,0.151437,False,[]
8,FIXATION,1,574.121,994.203,420.082,131.539457,4.252783,1.608891,58.416090,15.540025,...,471.7895,385.6253,603.2771,381.9321,"(536.4606573459716, 389.4191696682464)","(42.22665117615512, 5.040372634350404)",4.984848,2.008756,False,[]
9,SACCADE,2,996.204,1026.213,30.009,92.361556,2.986823,349.057322,130.673598,60.739930,...,604.4536,382.5109,695.1358,400.0436,"(664.08105, 391.42693125)","(35.79508118022293, 6.032089245103927)",3.664194,1.405535,False,[]


## Putting it all together

The same four steps, compactly — this is the pattern the rest of the guide builds on:

In [7]:
d = _helpers.load_example_trial()
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
labels, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
events = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
peyes.summarize_events(events).head(10)

,event_type,label,start_time,end_time,duration,distance,amplitude,azimuth,peak_velocity,median_velocity,...,start_x,start_y,end_x,end_y,center_pixel,pixel_std,dispersion,ellipse_area,is_outlier,outlier_reasons
0,FIXATION,1,6.000,210.046,204.046,7.356845,0.237962,99.967557,42.993417,13.650236,...,522.4870,423.0247,521.2136,415.7789,"(520.6592281553399, 418.31843883495134)","(1.3781321854123456, 2.85209201717164)",0.522272,0.049609,False,[]
1,SACCADE,2,212.047,272.057,60.010,241.146818,7.788051,178.513830,158.498845,79.263708,...,522.7053,416.1030,281.6396,409.8487,"(347.81459677419355, 412.33902903225817)","(85.4484891508158, 3.520229042144439)",8.706070,2.715996,False,[]
2,FIXATION,1,274.055,496.099,222.044,40.102058,1.297074,6.302341,43.679851,13.020898,...,278.8346,410.6481,318.6943,406.2459,"(291.6303732142857, 410.28303482142854)","(12.000146907694374, 1.5809510192619272)",1.601033,0.265221,False,[]
3,SACCADE,2,498.103,522.111,24.008,127.840747,4.133306,12.796403,148.080598,111.109023,...,319.6409,405.4971,444.3065,377.1820,"(381.0220769230769, 388.2436153846154)","(48.09015486131078, 14.103464151043285)",5.139701,3.510692,False,[]
4,FIXATION,1,524.112,528.115,4.003,2.411057,0.077987,194.343443,26.145356,19.518542,...,445.7363,378.2661,443.4004,378.8634,"(444.44613333333336, 378.6637)","(0.9691638813373706, 0.2811466165544314)",0.094876,0.001146,True,[min_duration]
5,SACCADE,2,530.112,534.112,4.000,5.273568,0.170577,197.506920,45.315013,40.785199,...,441.5088,379.7703,436.4795,381.3567,"(438.9026333333333, 380.62416666666667)","(2.057278221880132, 0.6533031829777594)",0.213989,0.006556,True,[min_duration]
6,FIXATION,1,536.112,542.110,5.998,2.191096,0.070873,197.182899,17.203618,17.179334,...,433.9817,382.3015,431.8884,382.9488,"(432.74295, 382.657425)","(0.8059122362267493, 0.3629671567442555)",0.093314,0.001362,True,[min_duration]
7,SACCADE,2,544.112,572.120,28.008,34.249128,1.107778,356.151584,74.872314,38.087965,...,433.7982,381.4550,467.9701,383.7537,"(452.6205333333334, 383.62452666666667)","(10.597112155278692, 1.7278169476603162)",1.279730,0.151437,False,[]
8,FIXATION,1,574.121,994.203,420.082,131.539457,4.252783,1.608891,58.416090,15.540025,...,471.7895,385.6253,603.2771,381.9321,"(536.4606573459716, 389.4191696682464)","(42.22665117615512, 5.040372634350404)",4.984848,2.008756,False,[]
9,SACCADE,2,996.204,1026.213,30.009,92.361556,2.986823,349.057322,130.673598,60.739930,...,604.4536,382.5109,695.1358,400.0436,"(664.08105, 391.42693125)","(35.79508118022293, 6.032089245103927)",3.664194,1.405535,False,[]


## What's next

- **[2 Datasets](./2%20Datasets.ipynb)** — the built-in datasets `load_example_trial` draws from, and how to load
  one yourself.
- **[3 Parsing Custom Data & Configuration](./3%20Parsing%20Custom%20Data%20%26%20Configuration.ipynb)** — using
  your own recordings instead of a built-in dataset.
- **[4 Detection Algorithms](./4%20Detection%20Algorithms.ipynb)** — the other detection algorithms besides Engbert,
  and their parameters.